# Real-Time Streaming Pipeline – Interactive Exploration Notebook

**Repository**: [sonofsparda24/Real-Time-Streaming-Pipeline](https://github.com/sonofsparda24/Real-Time-Streaming-Pipeline)
**Author**: Youssef Nahdi + Grok (xAI)
**Last updated**: December 2025

This notebook lets you explore every part of the pipeline hands-on:
- Generate fake logs
- Watch them flow through Kafka
- Run a live Spark Structured Streaming job
- Query Elasticsearch & HDFS
- Visualize results

Run the cells in order!

## 1. Start the Full Stack (Docker Compose)

In [ ]:
!cd .. && docker-compose up -d
!echo "Waiting 45 seconds for services to be healthy..."
!sleep 45
!docker-compose ps

## 2. Generate Fake Syslog Traffic

In [ ]:
import socket
import random
import time
from datetime import datetime

def send_syslog(message, host="localhost", port=514):
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    syslog_msg = f"<14>{datetime.now():%b %d %H:%M:%S} {random.choice(['web1', 'db1', 'app2', 'auth1'])} {message}\n"
    sock.sendto(syslog_msg.encode(), (host, port))
    sock.close()
    print(f"Sent: {syslog_msg.strip()}")

messages = [
    "kernel: INFO System booted successfully",
    "sshd[1234]: WARNING Accepted publickey for admin",
    "sshd[5678]: ERROR Failed password for invalid user",
    "myapp: CRITICAL Database connection lost",
    "nginx: INFO 200 GET /api/health",
    "auth-service: ERROR failed login attempt from 192.168.10.45"
]

print("Sending 20 log lines...")
for _ in range(20):
    send_syslog(random.choice(messages))
    time.sleep(0.3)

## 3. Peek Inside Kafka Topic

In [ ]:
!docker exec -it kafka kafka-console-consumer.sh \
    --bootstrap-server localhost:9092 \
    --topic syslog-stream \
    --from-beginning \
    --max-messages 10

## 4. Live Spark Structured Streaming (Mini Version)
Runs directly in the notebook – no submission needed.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (SparkSession.builder
         .master("local[*]")
         .appName("NotebookStreaming")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
         .getOrCreate())

# Simple syslog parsing schema (good enough for demo)
schema = "timestamp STRING, host STRING, message STRING"

raw = (spark
       .readStream
       .format("kafka")
       .option("kafka.bootstrap.servers", "localhost:9092")
       .option("subscribe", "syslog-stream")
       .option("startingOffsets", "latest")
       .load())

parsed = (raw
          .selectExpr("CAST(value AS STRING) as raw_line")
          .select(split(col("raw_line"), " ", 5).alias("fields"))
          .select(
              concat(col("fields")[0], lit(" "), col("fields")[1], lit(" "), col("fields")[2]).alias("timestamp"),
              col("fields")[3].alias("host"),
              regexp_extract(col("raw_line"), r"\]: (.*)", 1).alias("message"),
              when(col("raw_line").contains("ERROR"), "ERROR")
              .when(col("raw_line").contains("CRITICAL"), "CRITICAL")
              .when(col("raw_line").contains("WARNING"), "WARNING")
              .otherwise("INFO").alias("level")
          )
          .withColumn("processed_at", current_timestamp()))

# Console output (real job would write to ES + HDFS)
query = (parsed
         .writeStream
         .outputMode("append")
         .format("console")
         .option("truncate", False)
         .start())

print("Streaming started – generate more logs to see them appear live!")
query.awaitTermination(60)  # runs for 60 seconds
query.stop()

## 5. Query Elasticsearch (Fast Search)

In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd

es = Elasticsearch("http://localhost:9200")
res = es.search(index="logs-*", size=10, query={"match_all": {}})

hits = [hit["_source"] for hit in res["hits"]["hits"]]
df_es = pd.DataFrame(hits)
df_es[["@timestamp", "host", "level", "message"]].sort_values("@timestamp", ascending=False)

## 6. Query HDFS Parquet Files (Long-term Storage)

In [ ]:
df_hdfs = spark.read.parquet("hdfs://namenode:9000/logs")
print(f"Total historical logs in HDFS: {df_hdfs.count():,}")
df_hdfs.groupBy("level").count().orderBy(col("count").desc()).show()

## 7. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

level_counts = df_hdfs.groupBy("level").count().toPandas()
plt.figure(figsize=(8,5))
plt.bar(level_counts["level"], level_counts["count"], color=["green","orange","red","purple"])
plt.title("Log Level Distribution (HDFS)")
plt.xlabel("Level")
plt.ylabel("Count")
plt.show()

## 8. Run the Real Alert Consumer

In [ ]:
!python ../alerts/consume_alerts.py

## Done!

You just went end-to-end:
Generated logs → Kafka → Spark Streaming → Elasticsearch + HDFS → Kibana ready

Open Kibana → http://localhost:5601 to create your first dashboard.

To stop everything:
```bash
cd .. && docker-compose down
```